# SentinelAI -- Phase I
## openFDA MAUDE API: Pulling, Understanding, and Preparing Data

**Goal of this notebook:**
- Explore the openFDA API
- Pull MAUDE adverse events for a specific device type
- EDA: structure, fields, class distribution
- Save dataset for ML

**Docs:** https://open.fda.gov/apis/device/event/


---
## 📖 How to Read This Notebook (Beginner Guide)

This is **Notebook 01** — the starting point of the SentinelAI project.
Its job is to download raw adverse event data from the FDA and understand what it looks like.

### Key concepts explained here

**MAUDE** (Manufacturer and User Facility Device Experience)
: The FDA's public database of medical device problems, injuries, and malfunctions.
  Every serious incident must be reported. Think of it as a mandatory incident log
  for medical devices, going back to the 1990s.

**openFDA API**
: A public web interface that lets you query the MAUDE database programmatically
  (i.e. from Python code, without using a browser). API = Application Programming Interface.
  You send an HTTP request (like a web browser fetching a page) and get back JSON data.

**JSON** (JavaScript Object Notation)
: A structured text format for data exchange. Looks like nested Python dicts/lists.
  The MAUDE API returns records as deeply nested JSON — that is why we need `flatten_maude_record()`.

**Pagination**
: When a database has millions of records, the API does not return them all at once.
  It returns them in "pages" (e.g. 100 records at a time). You request page 1, then page 2, etc.
  The openFDA hard limit is 10,000 records per query (skip + limit <= 10,000).

**DataFrame** (pandas)
: A table-like data structure in Python (like an Excel spreadsheet in memory).
  Rows = individual MAUDE reports. Columns = fields (date, device name, narrative text, etc.).

**EDA** (Exploratory Data Analysis)
: Looking at the data before modelling — understanding distributions, missing values,
  outliers. "What do we have? What shape is it in?"

**Parquet**
: A compressed binary file format for storing DataFrames efficiently.
  Much smaller and faster to read than CSV for large datasets.

### What this notebook produces
- `data/raw/maude_pass1.parquet` — raw MAUDE records for insulin pumps (early exploration version)
- Understanding of the API structure → feeds into `src/vigilex/data/maude_client.py`
---

In [ ]:
import requests
import pandas as pd
import json
import time
import seaborn as sn
import matplotlib.pyplot as plt
from pathlib import Path

# Create output directories
Path('data').mkdir(exist_ok=True)
Path('data/raw').mkdir(exist_ok=True)

BASE_URL = 'https://api.fda.gov/device/event.json'
print('Setup OK')


## 1. Understanding the API Structure

The openFDA API has 3 main parameters:
- `search`: filter query (Lucene syntax)
- `count`: aggregate by field
- `limit` / `skip`: pagination (max. 1000 per request)


In [ ]:
# --- Step 1: How many entries exist for insulin pumps? ---
r = requests.get(BASE_URL, params={
    'search': 'device.generic_name:"insulin pump"',
    'limit': 1
})
total = r.json()['meta']['results']['total']
print(f'Total insulin pump events: {total:,}')


In [ ]:
# --- Step 2: Distribution of event types ---
r = requests.get(BASE_URL, params={
    'search': 'device.generic_name:"insulin pump"',
    'count': 'event_type.exact',
})
event_types = pd.DataFrame(r.json()['results'])
print('Event Types:')
print(event_types)


In [ ]:
# --- Step 3: Most frequent problem codes ---
r = requests.get(BASE_URL, params={
    'search': 'device.generic_name:"insulin pump"',
    'count': 'device.device_report_product_code.exact',
    'limit': 10
})
print('Top 10 Product Codes:')
print(pd.DataFrame(r.json()['results']))


In [ ]:
# --- Step 4: Temporal trend of reports ---
r = requests.get(BASE_URL, params={
    'search': 'device.generic_name:"insulin pump"',
    'count': 'date_of_event',
})



In [ ]:
df_trend = pd.DataFrame(r.json()['results'])
df_trend['date'] = pd.to_datetime(df_trend['time'], format='%Y%m%d')
df_trend = df_trend.set_index('date').sort_index()

df_yearly = df_trend['count'].resample('YE').sum().reset_index()
df_yearly.columns = ['date', 'count']
#df_trend.isna().sum()
df_yearly.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sn.lineplot(data=df_yearly, x='date', y='count', ax=ax)
ax.set_title('Insulin Pump Adverse Events per Year')
ax.set_ylabel('Number of Reports')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('data/trend_insulin_pumps.png', dpi=150)
plt.show()


## 2. Bulk Data Pull

openFDA allows max. **1000 records per request** and **1000 requests/day** without an API key.
With an API key (free, register at open.fda.gov) up to 120,000 requests/day are possible.

Strategy: pagination using the `skip` parameter.


In [ ]:
# --- API Key (optional but recommended) ---
# Register at: https://open.fda.gov/apis/authentication/
!pip install dotenv
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv('OPENFDA_API_KEY', '')
print('Key loaded:', bool(API_KEY))


In [ ]:
def fetch_maude_by_daterange(search_base, start_year=2015, end_year=2025, api_key=''):
    all_records = []

    for year in range(start_year, end_year + 1):
        # Space instead of + around TO keyword
        search = f'{search_base} AND date_of_event:[{year}0101 TO {year}1231]'
        skip = 0

        while True:
            params = {
                'search': search,
                'limit': 1000,
                'skip': skip
            }
            if api_key:
                params['api_key'] = api_key

            r = requests.get(BASE_URL, params=params, timeout=30)

            if r.status_code != 200:
                print(f'\n{year} Error: {r.status_code} -- {r.json().get("error", {}).get("message", "")}')
                break

            batch = r.json().get('results', [])
            if not batch:
                break

            all_records.extend(batch)
            skip += len(batch)
            print(f'{year} | {skip:>5} records...', end='\r')
            time.sleep(0.1)

        print(f'{year} | {skip:>5} records loaded')

    print(f'\nTotal: {len(all_records):,} records')
    return all_records


In [ ]:
# Einzeltest Jahr 2023
r = requests.get(BASE_URL, params={
    'search': 'device.generic_name:"insulin pump" AND date_of_event:[20230101 TO 20231231]',
    'limit': 1,
    'api_key': API_KEY
})
print(r.status_code)
print(r.json()['meta']['results']['total'])

In [ ]:
# Pull full dataset:
raw_data = fetch_maude_by_daterange(
    search_base='device.generic_name:"insulin pump"',
    start_year=2015,
    end_year=2025,
    api_key=API_KEY
)


## 3. JSON -> Flatten into DataFrame

MAUDE records are deeply nested. The following fields are relevant for ML:


In [ ]:
def flatten_maude_record(rec):
    device  = rec.get('device',  [{}])
    device  = device[0]  if device  else {}
    patient = rec.get('patient', [{}])
    patient = patient[0] if patient else {}
    mdr     = rec.get('mdr_text', [{}])
    mdr     = mdr[0]     if mdr     else {}

    outcome_list = patient.get('sequence_number_outcome', [])

    return {
        'report_number':     rec.get('report_number'),
        'date_received':     rec.get('date_received'),
        'date_of_event':     rec.get('date_of_event'),
        'generic_name':      device.get('generic_name'),
        'brand_name':        device.get('brand_name'),
        'manufacturer_name': device.get('manufacturer_d_name'),
        'device_age_text':   device.get('device_age_text'),
        'product_code':      device.get('device_report_product_code'),
        'implant_flag':      device.get('implant_flag'),
        'single_use_flag':   device.get('single_use_flag'),
        'event_type':        rec.get('event_type'),
        'adverse_event_flag':rec.get('adverse_event_flag'),
        'report_source':     rec.get('report_source_code'),
        'reporter_country':  rec.get('reporter_country_code'),
        'patient_outcome':   outcome_list[0] if outcome_list else None,
        'narrative_text':    mdr.get('text', ''),
    }



In [ ]:
df = pd.DataFrame([flatten_maude_record(r) for r in raw_data])
df.to_parquet('data/raw/maude_pass1.parquet', index=False)
import os
os.chdir('..')  # vigilex/notebooks -> vigilex
Path('data/raw').mkdir(parents=True, exist_ok=True)
df.to_parquet('data/raw/maude_pass1.parquet', index=False)
print('Saved:', Path('data/raw/maude_pass1.parquet').stat().st_size / 1e6, 'MB')


In [ ]:
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# --- Quick EDA ---
print('=== Dtypes ===')
print(df.dtypes)

print('\n=== Missing Values (%) ===')
print((df.isnull().mean() * 100).round(1).sort_values(ascending=False))

print('\n=== event_type Distribution (ML Target 1) ===')
print(df['event_type'].value_counts())

print('\n=== patient_outcome Distribution (ML Target 2) ===')
print(df['patient_outcome'].value_counts())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Event Types
df['event_type'].value_counts().plot(
    kind='bar', ax=axes[0], color='steelblue', title='Event Type Distribution'
)
axes[0].set_xlabel('')

# Patient Outcomes
df['patient_outcome'].value_counts().head(8).plot(
    kind='bar', ax=axes[1], color='coral', title='Patient Outcome Distribution'
)
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig('data/eda_distributions.png', dpi=150)
plt.show()


In [ ]:
# --- Top 10 manufacturers ---
df['manufacturer_name'].value_counts().head(10).plot(
    kind='barh', figsize=(10, 5),
    title='Top 10 Manufacturers by Report Count',
    color='steelblue'
)
plt.tight_layout()
plt.show()

print('\nTop manufacturers:')
print(df['manufacturer_name'].value_counts().head(10))


## 4. Saving the Data

Save raw data as Parquet (efficient) and CSV (human-readable).


In [ ]:
# Parse date columns
for col in ['date_received', 'date_of_event']:
    df[col] = pd.to_datetime(df[col], format='%Y%m%d', errors='coerce')

# Save
df.to_parquet('data/raw/maude_insulin_pumps.parquet', index=False)
df.to_csv('data/raw/maude_insulin_pumps.csv', index=False)

# Raw JSON as backup
with open('data/raw/maude_raw.json', 'w') as f:
    json.dump(raw_data, f)

print(f'Saved: {len(df):,} records')
print(f'Parquet: data/raw/maude_insulin_pumps.parquet')
print(f'CSV:     data/raw/maude_insulin_pumps.csv')


## 5. Crossing with the Recall Database (ML Target: Recall Yes/No)

The FDA Recall Database is also available via openFDA.
This is used to assign labels: did a device get recalled after X complaints?


In [ ]:
RECALL_URL = 'https://api.fda.gov/device/recall.json'

# Fetch recalls for insulin pumps
r = requests.get(RECALL_URL, params={
    'search': 'product_description:"insulin pump"',
    'limit': 100
})

if r.status_code == 200:
    recalls = pd.DataFrame(r.json()['results'])
    print(f'Recalls found: {len(recalls)}')
    print(recalls[['recall_number', 'product_description',
                   'recalling_firm', 'recall_initiation_date',
                   'classification']].head(5))
else:
    print(f'Error: {r.status_code}')


In [ ]:
# Recall class definitions:
# Class I   = highest risk (life-threatening)
# Class II  = moderate risk (temporary health impairment)
# Class III = lowest risk (no health risk)

if 'recalls' in dir() and len(recalls) > 0:
    print('Recall classes (same classification logic as SAE severity):')
    print(recalls['classification'].value_counts())

    # Manufacturers with recalls
    print('\nManufacturers with recalls:')
    print(recalls['recalling_firm'].value_counts().head(10))


## Next Steps (Notebook 02)

1. Build features: rolling-window complaint counts, severity score
2. Assign targets: recall within 12 months after first complaint peak
3. Baseline models: LogReg, RandomForest
4. Evaluation: recall-focused (no FN risk in patient safety context)

**Note:** Your DKFZ background directly helps with feature interpretation --
you know which complaint patterns indicate a real problem in practice.
